# Problema con sistema de tipo 1

In [1]:
import control as ctrl
import numpy as np
import matplotlib.pyplot as plt

Para el sistema :

$$ G(s) = \frac{10}{s\left(\frac{s}{2.5}+1\right)\left(\frac{s}{6}+1\right)} $$

Se requiere un sistema que tenga un margen de fase de 45 grados y una constante de velocidad $K_v=10$

In [2]:
s=ctrl.tf('s')
G1=10/((s/2.5+1)*(s/6+1)*s)

Analizamos los polos del sistema a lazo abierto

In [3]:
G1.pole()

AttributeError: 'TransferFunction' object has no attribute 'pole'

In [ ]:
ctrl.bode(G1, dB=True);
plt.gcf().set_size_inches(12,8)

In [ ]:
_,pm,_,_,wp,_=ctrl.stability_margins(G1)
print(pm,wp)

In [ ]:
ctrl.dcgain(ctrl.minreal(G1*s)) # el minreal es para simplificar el pole en cero y pueda evaluarla

In [ ]:
phi_max=(45+10-pm)
print(f"El ángulo máximo a agregar es {phi_max}")
phi_max = phi_max*np.pi/180
alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
print(f"Esto produce un esto pruduce una relación z/p de {alpha}")

Probamos poniendo el el $\omega_{max}$ en la frecuencia de corte.

In [ ]:
wmax=wp
# phi_max=85

phi_max = phi_max*np.pi/180
alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
alpha

In [ ]:
TD=1/(wmax*np.sqrt(alpha))

In [ ]:
z=-1/TD
p=-1/(alpha*TD)
print(z, p)

In [ ]:
Dc=ctrl.tf([-1/z,1],[-1/p,1])
ctrl.stability_margins(G1*Dc)

Vemos que esto no sirve por que la curva de fase se baja demasiado. Deberíamos rediseñar.

In [ ]:
wmax=12
phi_max=60*(np.pi)/180

alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
TD=1/(wmax*np.sqrt(alpha))
z=-1/TD
p=z/alpha

Dc=ctrl.tf([-1/z,1],[-1/p,1])
print(Dc.zero(), Dc.pole(), Dc.dcgain(), alpha)

In [ ]:
ctrl.stability_margins(G1*Dc)

Vemos que no se puede llegar a los 45 grados de margen de fase con solo un compensador de adelanto.

Voy a probar con dos dejando el siguiente compensador:

In [ ]:
wmax=9
phi_max=35*(np.pi)/180
alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
TD=1/(wmax*np.sqrt(alpha))
z=1/TD
p=z/alpha

Dc1=ctrl.tf([1/z,1],[1/p,1])
print(Dc1.zero(), Dc1.pole(), Dc1.dcgain(), alpha)

Probamos con la misma idea:

In [ ]:
wmax=10
phi_max=40*(np.pi)/180
alpha = (1-np.sin(phi_max))/(1+np.sin(phi_max))
TD=1/(wmax*np.sqrt(alpha))
z=1/TD
p=z/alpha


Dc2=ctrl.tf([1/z,1],[1/p,1])
print(Dc2.zero(), Dc2.pole(), Dc2.dcgain(), alpha)

In [ ]:
ctrl.stability_margins(G1*Dc1*Dc2)

In [ ]:
T=ctrl.feedback(G1*Dc1*Dc2)
t,y = ctrl.step_response(T, T=np.linspace(0,2,2401))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t,y)
ax.set_title('Respuesta al escalón')
ax.set_xlabel('Tiempo')
ax.grid()

In [ ]:
T=ctrl.feedback(G1*Dc1*Dc2)
t,y = ctrl.step_response(T*ctrl.tf(1,[1,0]), T=np.linspace(0,500,2401))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t,y)
ax.set_title('Respuesta al escalón')
ax.set_xlabel('Tiempo')
ax.grid()

In [ ]:
error = y[-1]-t[-1]
print(error)

In [ ]:
(Dc1*Dc2).pole()

In [ ]:
(Dc1*Dc2).zero()